# 🔴 Real-Time Data Analysis Project
### End-to-End Pipeline: Ingestion → Cleaning → EDA → Modeling → Live Dashboard → Deploy

---
**Project Overview:**
- Step 1: Project Setup & Library Installation
- Step 2: Data Collection & Ingestion (Simulated Stream)
- Step 3: Data Cleaning & Preprocessing
- Step 4: Exploratory Data Analysis (EDA)
- Step 5: Feature Engineering
- Step 6: Machine Learning Modeling
- Step 7: Model Evaluation & Comparison
- Step 8: Anomaly Detection
- Step 9: Real-Time Dashboard (Plotly)
- Step 10: Save Artifacts & Export

---
## ⚙️ Step 1: Project Setup & Library Installation

In [1]:
# Install required libraries (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn plotly joblib scipy

# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# Utilities
import warnings
import datetime
import random
import json
import os
import joblib
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries loaded successfully!')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')
print(f'   Sklearn : {__import__("sklearn").__version__}')

ModuleNotFoundError: No module named 'pandas'

---
## 📡 Step 2: Data Collection & Ingestion (Simulated Real-Time Stream)

In [ ]:
# Simulate a real-time IoT sensor data stream
# In production: replace with Kafka consumer / REST API / WebSocket

def generate_sensor_stream(n_records=5000):
    """
    Simulate a real-time sensor data stream.
    Mimics: temperature sensor, humidity sensor, pressure sensor.
    """
    start_time = datetime.datetime(2024, 1, 1, 0, 0, 0)
    timestamps = [start_time + datetime.timedelta(seconds=i*30) for i in range(n_records)]

    # Simulate sensor readings with natural patterns + noise
    hour_of_day = np.array([t.hour for t in timestamps])
    day_of_week = np.array([t.weekday() for t in timestamps])

    # Temperature: peaks at 2PM, cooler at night
    temp_base = 25 + 8 * np.sin((hour_of_day - 6) * np.pi / 12)
    temperature = temp_base + np.random.normal(0, 1.5, n_records)

    # Humidity: inversely related to temperature
    humidity = 80 - 0.8 * (temperature - 20) + np.random.normal(0, 3, n_records)
    humidity = np.clip(humidity, 20, 100)

    # Pressure: slow drift with random walk
    pressure = 1013 + np.cumsum(np.random.normal(0, 0.05, n_records))
    pressure = np.clip(pressure, 990, 1040)

    # Energy consumption: target variable
    energy = (0.5 * temperature + 0.3 * humidity +
              0.1 * (day_of_week < 5).astype(int) * 10 +
              np.random.normal(0, 2, n_records))

    # Inject missing values (~3%)
    for col_arr in [temperature, humidity, pressure, energy]:
        mask = np.random.rand(n_records) < 0.03
        col_arr[mask] = np.nan

    # Inject outliers (~1%)
    outlier_idx = np.random.choice(n_records, size=int(n_records * 0.01), replace=False)
    temperature[outlier_idx] += np.random.choice([-20, 20], len(outlier_idx))

    # Inject duplicate rows (~2%)
    dup_indices = np.random.choice(n_records, size=int(n_records * 0.02))

    sensor_ids = np.random.choice(['SENSOR_A', 'SENSOR_B', 'SENSOR_C'], n_records)
    locations  = np.where(np.isin(sensor_ids, ['SENSOR_A']), 'Zone_1',
                 np.where(np.isin(sensor_ids, ['SENSOR_B']), 'Zone_2', 'Zone_3'))

    df = pd.DataFrame({
        'timestamp'  : timestamps,
        'sensor_id'  : sensor_ids,
        'location'   : locations,
        'temperature': temperature,
        'humidity'   : humidity,
        'pressure'   : pressure,
        'energy_kwh' : energy
    })

    # Add duplicates
    df = pd.concat([df, df.iloc[dup_indices]], ignore_index=True)

    return df

# Generate dataset
df_raw = generate_sensor_stream(n_records=5000)

print(f'📦 Raw dataset shape: {df_raw.shape}')
print(f'📅 Date range: {df_raw.timestamp.min()} → {df_raw.timestamp.max()}')
print()
print(df_raw.head())

In [ ]:
# Dataset info and missing values
print('=== DATASET INFO ===')
df_raw.info()
print()
print('=== MISSING VALUES ===')
print(df_raw.isnull().sum())
print()
print('=== BASIC STATISTICS ===')
df_raw.describe().round(2)

---
## 🧹 Step 3: Data Cleaning & Preprocessing

In [ ]:
def clean_data(df):
    df = df.copy()
    original_shape = df.shape

    # 1. Remove duplicates
    df = df.drop_duplicates()
    print(f'✅ Duplicates removed : {original_shape[0] - df.shape[0]} rows')

    # 2. Sort by timestamp
    df = df.sort_values('timestamp').reset_index(drop=True)

    # 3. Fill missing values
    for col in ['temperature', 'humidity', 'pressure', 'energy_kwh']:
        before = df[col].isnull().sum()
        df[col] = df[col].fillna(df[col].median())
        print(f'✅ {col:12s} : {before} NaNs filled with median ({df[col].median():.2f})')

    # 4. Remove outliers using IQR method
    before_outlier = df.shape[0]
    for col in ['temperature', 'humidity', 'pressure', 'energy_kwh']:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        df  = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]
    print(f'✅ Outliers removed   : {before_outlier - df.shape[0]} rows')

    # 5. Clip physical sensor bounds
    df['temperature'] = df['temperature'].clip(-10, 55)
    df['humidity']    = df['humidity'].clip(0, 100)
    df['pressure']    = df['pressure'].clip(970, 1050)

    df = df.reset_index(drop=True)
    print(f'\n📊 Final shape: {original_shape} → {df.shape}')
    return df

df_clean = clean_data(df_raw)
print('\n', df_clean.describe().round(2))

In [ ]:
# Visualize before vs after cleaning
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cols = ['temperature', 'humidity', 'pressure', 'energy_kwh']
colors_raw   = '#E24B4A'
colors_clean = '#185FA5'

for ax, col in zip(axes.flatten(), cols):
    ax.hist(df_raw[col].dropna(),   bins=60, alpha=0.5, color=colors_raw,   label='Raw',   density=True)
    ax.hist(df_clean[col].dropna(), bins=60, alpha=0.7, color=colors_clean, label='Clean', density=True)
    ax.set_title(col.replace('_', ' ').title(), fontsize=13)
    ax.legend(fontsize=10)
    ax.set_ylabel('Density')

plt.suptitle('Before vs After Cleaning — Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('cleaning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: cleaning_comparison.png')

---
## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Summary stats
print('=== CLEANED DATA SUMMARY ===')
print(df_clean.describe().round(3))
print()
print('=== SENSOR DISTRIBUTION ===')
print(df_clean['sensor_id'].value_counts())
print()
print('=== LOCATION DISTRIBUTION ===')
print(df_clean['location'].value_counts())

In [ ]:
# Time series plot — first 7 days
df_week = df_clean[df_clean['timestamp'] < df_clean['timestamp'].min() + datetime.timedelta(days=7)]

fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
metrics = ['temperature', 'humidity', 'pressure', 'energy_kwh']
palette = ['#E24B4A', '#185FA5', '#639922', '#854F0B']

for ax, metric, color in zip(axes, metrics, palette):
    ax.plot(df_week['timestamp'], df_week[metric], color=color, linewidth=0.8, alpha=0.8)
    ax.set_ylabel(metric.replace('_',' ').title(), fontsize=11)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.suptitle('Real-Time Sensor Data — First 7 Days', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('timeseries_7days.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: timeseries_7days.png')

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

num_cols = ['temperature', 'humidity', 'pressure', 'energy_kwh']
corr = df_clean[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=axes[0], linewidths=0.5,
            annot_kws={'size': 12})
axes[0].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')

# Scatter: temp vs energy
scatter_colors = {'Zone_1': '#E24B4A', 'Zone_2': '#185FA5', 'Zone_3': '#639922'}
for zone, color in scatter_colors.items():
    mask_z = df_clean['location'] == zone
    axes[1].scatter(df_clean.loc[mask_z, 'temperature'],
                    df_clean.loc[mask_z, 'energy_kwh'],
                    c=color, alpha=0.3, s=10, label=zone)
axes[1].set_xlabel('Temperature (°C)', fontsize=11)
axes[1].set_ylabel('Energy (kWh)', fontsize=11)
axes[1].set_title('Temperature vs Energy by Zone', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: eda_correlation.png')

In [ ]:
# Hourly and weekly patterns
df_clean['hour']        = df_clean['timestamp'].dt.hour
df_clean['day_of_week'] = df_clean['timestamp'].dt.dayofweek
df_clean['month']       = df_clean['timestamp'].dt.month
df_clean['is_weekend']  = (df_clean['day_of_week'] >= 5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly = df_clean.groupby('hour')['energy_kwh'].mean()
axes[0].bar(hourly.index, hourly.values, color='#185FA5', alpha=0.85, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Hour of Day', fontsize=11)
axes[0].set_ylabel('Avg Energy (kWh)', fontsize=11)
axes[0].set_title('Average Energy Consumption by Hour', fontsize=13, fontweight='bold')
axes[0].axhline(hourly.mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {hourly.mean():.1f}')
axes[0].legend()

days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
daily = df_clean.groupby('day_of_week')['energy_kwh'].mean()
bar_colors = ['#185FA5']*5 + ['#E24B4A']*2
axes[1].bar(days, daily.values, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Day of Week', fontsize=11)
axes[1].set_ylabel('Avg Energy (kWh)', fontsize=11)
axes[1].set_title('Average Energy by Day (Red=Weekend)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: eda_patterns.png')

In [ ]:
# Box plots by location
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['temperature', 'humidity', 'energy_kwh']):
    df_clean.boxplot(column=col, by='location', ax=ax,
                     patch_artist=True,
                     boxprops=dict(facecolor='#B5D4F4', color='#185FA5'),
                     medianprops=dict(color='#E24B4A', linewidth=2))
    ax.set_title(col.replace('_',' ').title(), fontsize=12)
    ax.set_xlabel('Location')

plt.suptitle('Feature Distribution by Zone', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: eda_boxplots.png')

---
## ⚙️ Step 5: Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Time features
    df['hour']           = df['timestamp'].dt.hour
    df['day_of_week']    = df['timestamp'].dt.dayofweek
    df['month']          = df['timestamp'].dt.month
    df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
    df['is_peak_hour']   = df['hour'].isin(range(9, 18)).astype(int)
    df['hour_sin']       = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']       = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin']        = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos']        = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # Rolling statistics (window = 10 records)
    df = df.sort_values('timestamp').reset_index(drop=True)
    for col in ['temperature', 'humidity', 'energy_kwh']:
        df[f'{col}_roll_mean'] = df[col].rolling(window=10, min_periods=1).mean()
        df[f'{col}_roll_std']  = df[col].rolling(window=10, min_periods=1).std().fillna(0)

    # Lag features
    for lag in [1, 5, 10]:
        df[f'energy_lag_{lag}'] = df['energy_kwh'].shift(lag).fillna(method='bfill')

    # Interaction features
    df['heat_index']     = df['temperature'] + 0.33 * df['humidity'] - 4
    df['temp_humidity']  = df['temperature'] * df['humidity']

    # Encode categorical
    le = LabelEncoder()
    df['location_enc']   = le.fit_transform(df['location'])
    df['sensor_id_enc']  = le.fit_transform(df['sensor_id'])

    print(f'✅ Features created. Total columns: {df.shape[1]}')
    return df

df_feat = engineer_features(df_clean)

feature_cols = [c for c in df_feat.columns
                if c not in ['timestamp','sensor_id','location','energy_kwh']]
print(f'\n📌 Feature columns ({len(feature_cols)}):')
print(feature_cols)

In [ ]:
# Feature correlation with target
corr_with_target = df_feat[feature_cols + ['energy_kwh']].corr()['energy_kwh'].drop('energy_kwh')
corr_sorted = corr_with_target.abs().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))
colors_bar = ['#E24B4A' if v < 0 else '#185FA5' for v in corr_with_target[corr_sorted.index]]
ax.barh(corr_sorted.index, corr_with_target[corr_sorted.index], color=colors_bar, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlabel('Pearson Correlation with Energy (kWh)', fontsize=11)
ax.set_title('Feature Correlation with Target Variable', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: feature_correlation.png')

---
## 🤖 Step 6: Machine Learning Modeling

In [ ]:
# Prepare X, y
X = df_feat[feature_cols].select_dtypes(include=[np.number])
y = df_feat['energy_kwh']

X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train size : {X_train.shape}')
print(f'✅ Test size  : {X_test.shape}')
print(f'✅ Features   : {X.shape[1]}')

In [ ]:
# Train multiple models
models = {
    'Linear Regression'    : LinearRegression(),
    'Decision Tree'        : DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest'        : RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
}

results = {}
trained_models = {}

print(f'{"Model":<25} {"R²":>8} {"RMSE":>8} {"MAE":>8}')
print('-' * 55)

for name, model in models.items():
    if name == 'Linear Regression':
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    r2   = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)

    results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'predictions': preds}
    trained_models[name] = model

    print(f'{name:<25} {r2:>8.3f} {rmse:>8.3f} {mae:>8.3f}')

best_model_name = max(results, key=lambda k: results[k]['R2'])
print(f'\n🏆 Best model: {best_model_name} (R²={results[best_model_name]["R2"]:.3f})')

---
## 📊 Step 7: Model Evaluation & Visualization

In [ ]:
# Compare model performance
model_names = list(results.keys())
r2_vals   = [results[m]['R2']   for m in model_names]
rmse_vals = [results[m]['RMSE'] for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bar_colors = ['#E24B4A' if m != best_model_name else '#185FA5' for m in model_names]
axes[0].barh(model_names, r2_vals, color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('R² Score', fontsize=11)
axes[0].set_title('R² Score Comparison (higher = better)', fontsize=12, fontweight='bold')
axes[0].axvline(0.9, linestyle='--', color='green', alpha=0.6, label='0.9 threshold')
axes[0].legend()
for i, v in enumerate(r2_vals):
    axes[0].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=10)

axes[1].barh(model_names, rmse_vals, color=bar_colors[::-1], alpha=0.85, edgecolor='white')
axes[1].set_xlabel('RMSE', fontsize=11)
axes[1].set_title('RMSE Comparison (lower = better)', fontsize=12, fontweight='bold')
for i, v in enumerate(rmse_vals):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: model_comparison.png')

In [ ]:
# Actual vs Predicted — best model
best_preds = results[best_model_name]['predictions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, best_preds, alpha=0.3, s=10, color='#185FA5')
lim = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Energy (kWh)', fontsize=11)
axes[0].set_ylabel('Predicted Energy (kWh)', fontsize=11)
axes[0].set_title(f'{best_model_name}\nActual vs Predicted', fontsize=12, fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test.values - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.3, s=10, color='#854F0B')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('Residuals', fontsize=11)
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: actual_vs_predicted.png')

In [ ]:
# Feature importance (Random Forest)
rf_model = trained_models['Random Forest']
imp = pd.Series(rf_model.feature_importances_, index=X_train.columns)
imp = imp.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
imp[::-1].plot(kind='barh', ax=ax, color='#185FA5', alpha=0.85, edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: feature_importance.png')

---
## 🚨 Step 8: Anomaly Detection

In [ ]:
from sklearn.ensemble import IsolationForest

# Isolation Forest anomaly detection
iso = IsolationForest(contamination=0.03, random_state=42, n_jobs=-1)
anomaly_features = df_clean[['temperature','humidity','pressure','energy_kwh']].fillna(0)
df_clean['anomaly'] = iso.fit_predict(anomaly_features)
df_clean['anomaly_label'] = df_clean['anomaly'].map({1: 'Normal', -1: 'Anomaly'})

n_anomalies = (df_clean['anomaly'] == -1).sum()
print(f'🚨 Anomalies detected : {n_anomalies} ({n_anomalies/len(df_clean)*100:.1f}%)')
print(f'✅ Normal points       : {(df_clean["anomaly"]==1).sum()}')

# Visualize anomalies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {'Normal': '#185FA5', 'Anomaly': '#E24B4A'}
for label, color in colors_map.items():
    mask = df_clean['anomaly_label'] == label
    axes[0].scatter(df_clean.loc[mask, 'temperature'],
                    df_clean.loc[mask, 'energy_kwh'],
                    c=color, alpha=0.4, s=12 if label=='Normal' else 40,
                    label=label, zorder=2 if label=='Anomaly' else 1)
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Energy (kWh)')
axes[0].set_title('Anomaly Detection — Temp vs Energy', fontsize=12, fontweight='bold')
axes[0].legend()

# Time series with anomalies highlighted
sample = df_clean.iloc[:500]
axes[1].plot(sample['timestamp'], sample['energy_kwh'], color='#185FA5', linewidth=0.8, alpha=0.7, label='Normal')
anom = sample[sample['anomaly'] == -1]
axes[1].scatter(anom['timestamp'], anom['energy_kwh'],
                color='#E24B4A', s=50, zorder=5, label='Anomaly', marker='x')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Energy (kWh)')
axes[1].set_title('Anomalies in Time Series (first 500 pts)', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: anomaly_detection.png')

---
## 📈 Step 9: Real-Time Dashboard (Interactive Plotly)

In [ ]:
# Interactive Plotly Dashboard
df_dash = df_clean.sort_values('timestamp').copy()
df_dash_sample = df_dash.iloc[:500]  # first 500 for speed

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Energy Consumption Over Time',
        'Temperature & Humidity',
        'Hourly Energy Pattern',
        'Energy Distribution by Zone',
        'Correlation: Temp vs Energy',
        'Anomaly Map'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Row 1: Energy time series
fig.add_trace(go.Scatter(
    x=df_dash_sample['timestamp'], y=df_dash_sample['energy_kwh'],
    mode='lines', name='Energy', line=dict(color='#185FA5', width=1.5)
), row=1, col=1)

# Row 1: Temp + Humidity
fig.add_trace(go.Scatter(
    x=df_dash_sample['timestamp'], y=df_dash_sample['temperature'],
    mode='lines', name='Temp (°C)', line=dict(color='#E24B4A', width=1.5)
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=df_dash_sample['timestamp'], y=df_dash_sample['humidity'],
    mode='lines', name='Humidity (%)', line=dict(color='#639922', width=1.5),
    yaxis='y4'
), row=1, col=2)

# Row 2: Hourly pattern
hourly = df_dash.groupby('hour')['energy_kwh'].mean().reset_index()
fig.add_trace(go.Bar(
    x=hourly['hour'], y=hourly['energy_kwh'],
    name='Hourly Avg', marker_color='#854F0B'
), row=2, col=1)

# Row 2: Box by zone
for zone, color in [('Zone_1','#E24B4A'),('Zone_2','#185FA5'),('Zone_3','#639922')]:
    fig.add_trace(go.Box(
        y=df_dash[df_dash['location']==zone]['energy_kwh'],
        name=zone, marker_color=color
    ), row=2, col=2)

# Row 3: Scatter temp vs energy
fig.add_trace(go.Scatter(
    x=df_dash['temperature'], y=df_dash['energy_kwh'],
    mode='markers', name='Temp vs Energy',
    marker=dict(color='#185FA5', opacity=0.3, size=4)
), row=3, col=1)

# Row 3: Anomalies
normal = df_dash[df_dash['anomaly']==1].iloc[:300]
anom   = df_dash[df_dash['anomaly']==-1]
fig.add_trace(go.Scatter(
    x=normal['temperature'], y=normal['pressure'],
    mode='markers', name='Normal',
    marker=dict(color='#185FA5', size=4, opacity=0.4)
), row=3, col=2)
fig.add_trace(go.Scatter(
    x=anom['temperature'], y=anom['pressure'],
    mode='markers', name='Anomaly',
    marker=dict(color='#E24B4A', size=8, symbol='x')
), row=3, col=2)

fig.update_layout(
    height=900,
    title_text='Real-Time Sensor Analytics Dashboard',
    title_font_size=18,
    showlegend=True,
    template='plotly_white'
)
fig.write_html('realtime_dashboard.html')
fig.show()
print('📊 Dashboard saved: realtime_dashboard.html')

---
## 💾 Step 10: Save Artifacts & Export

In [ ]:
import os

os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Save best model
best_model_obj = trained_models[best_model_name]
joblib.dump(best_model_obj, f'models/{best_model_name.replace(" ","_")}.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

# Save processed data
df_feat.to_csv('data/processed_data.csv', index=False)
df_clean[df_clean['anomaly']==-1].to_csv('data/anomalies.csv', index=False)

# Save model metrics
metrics_df = pd.DataFrame([
    {'Model': m, 'R2': v['R2'], 'RMSE': v['RMSE'], 'MAE': v['MAE']}
    for m, v in results.items()
]).sort_values('R2', ascending=False)
metrics_df.to_csv('data/model_metrics.csv', index=False)

print('✅ Artifacts saved:')
print(f'   models/{best_model_name.replace(" ","_")}.pkl')
print('   models/scaler.pkl')
print('   data/processed_data.csv')
print('   data/anomalies.csv')
print('   data/model_metrics.csv')
print('   realtime_dashboard.html')
print()
print('=== FINAL MODEL METRICS ===')
print(metrics_df.to_string(index=False))

In [ ]:
# Predict on new real-time data point
def predict_realtime(temperature, humidity, pressure, hour, day_of_week,
                     is_weekend=0, location_enc=0, sensor_id_enc=0):
    """
    Predict energy consumption for a new real-time data point.
    """
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    day_sin  = np.sin(2 * np.pi * day_of_week / 7)
    day_cos  = np.cos(2 * np.pi * day_of_week / 7)
    heat_idx = temperature + 0.33 * humidity - 4
    temp_hum = temperature * humidity
    is_peak  = 1 if 9 <= hour <= 17 else 0

    # Build feature vector matching training columns
    feature_row = pd.DataFrame([{
        'temperature': temperature, 'humidity': humidity, 'pressure': pressure,
        'hour': hour, 'day_of_week': day_of_week, 'month': 1,
        'is_weekend': is_weekend, 'is_peak_hour': is_peak,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'day_sin': day_sin, 'day_cos': day_cos,
        'temperature_roll_mean': temperature, 'temperature_roll_std': 0,
        'humidity_roll_mean': humidity, 'humidity_roll_std': 0,
        'energy_kwh_roll_mean': 40, 'energy_kwh_roll_std': 5,
        'energy_lag_1': 40, 'energy_lag_5': 40, 'energy_lag_10': 40,
        'heat_index': heat_idx, 'temp_humidity': temp_hum,
        'location_enc': location_enc, 'sensor_id_enc': sensor_id_enc
    }])

    feature_row = feature_row[X_train.columns]
    prediction = trained_models['Random Forest'].predict(feature_row)[0]
    return round(prediction, 3)

# Example real-time prediction
pred = predict_realtime(
    temperature=32.5, humidity=65, pressure=1012,
    hour=14, day_of_week=1  # Tuesday 2PM
)
print(f'⚡ Predicted Energy Consumption: {pred} kWh')
print(f'   (Input: 32.5°C, 65% humidity, 1012 hPa, Tuesday 2PM)')

In [ ]:
# Final project summary
print('=' * 55)
print('       REAL-TIME DATA ANALYSIS PROJECT — SUMMARY')
print('=' * 55)
print(f'  Total raw records       : {df_raw.shape[0]:,}')
print(f'  Records after cleaning  : {df_clean.shape[0]:,}')
print(f'  Features engineered     : {len(feature_cols)}')
print(f'  Models trained          : {len(models)}')
print(f'  Best model              : {best_model_name}')
print(f'  Best R²                 : {results[best_model_name]["R2"]:.3f}')
print(f'  Best RMSE               : {results[best_model_name]["RMSE"]:.3f}')
print(f'  Anomalies detected      : {n_anomalies} ({n_anomalies/len(df_clean)*100:.1f}%)')
print('=' * 55)
print('  Outputs:')
print('    models/    — trained model + scaler')
print('    data/      — processed CSV + metrics')
print('    *.png      — all visualizations')
print('    realtime_dashboard.html — interactive viz')
print('=' * 55)